# Introduction - Query Model

In [ ]:
import pennylane as qp
import pennylane.numpy as np

# Deutsch's algorithm

Let's implemlement the following oracle given Figure 1 from the Seminar Handout:

$U_f \ket{x}\ket{y}=\ket{x}\ket{y\oplus f(x)}$

In [ ]:

dev = qp.device("default.qubit", wires=2)

def oracle(f):
    if f == "const_0":
        ##### TODO #####
    elif f == "const_1":
        ##### TODO #####
    elif f == "identity":        
        ##### TODO #####
    elif f == "negation":        
        ##### TODO #####

Here is the circuit:

![Deutsch circuit](figures/deutsch.png)

In [ ]:

@qp.qnode(dev)
def deutsch(f):
    ##### TODO #####
    return qp.probs(wires=0)

for f in ["const_0", "const_1", "identity", "negation"]:
    p = deutsch(f)
    print(f"{f:9s} P(0)={p[0]:.2f}  ->", "constant" if p[0] > 0.5 else "balanced")

In [ ]:
qp.draw_mpl(deutsch,style = "pennylane")("negation")

# Deutsch-Josza algorithm

Here is the circuit:

![Deutsch circuit](figures/dj.png)

In [ ]:
n_bits = 4
dev = qp.device("default.qubit", wires=n_bits)

We have a boolean function  $f:\{0,1\}^n\rightarrow\{0,1\}$ s.t.:
$$f(x) = 1 \iff x \in \texttt{combos}$$

In [ ]:
def multisol_combo(n_bits, how_many):
    """Return how_many distinct random bit strings of length n_bits."""
    indices = np.random.choice(2**n_bits, size=how_many, replace=False)
    return [[int(b) for b in np.binary_repr(i, width=n_bits)] for i in indices]

In [ ]:
multisol_combo(n_bits, 3)

### Let's create the oracle

For one solution we have $U_f=\text{Id}-2\ket{s}\bra{s}$ but for multiple ones: $$U_f = I - 2\sum_{s\in S}\ket{s}\bra{s}$$

In [ ]:
def multisol_oracle_matrix(combos):
    """
    Args:
        combos (list[list[int]]): A list of secret bit strings.

    Returns:
        array[float]: The matrix representation of the oracle.
    """
    # Create the identity matrix
    matrix = np.identity(2**n_bits)

    # convert each bit string in combos into its integer index, i.e. its position in the state vector or matrix.
    # hint: use "np.ravel_multi_index"
    ##### TODO #####

    # Apply the indices
    ##### TODO #####
    
    return matrix

Let's do the DJ circuit using pennylane's function ` qp.QubitUnitary(matrix, wires=range(n_bits))`. It allows to create a gate out of the matrix we've just built!

In [ ]:
@qp.qnode(dev)
def circuit(combos):
    """
    Args:
        combos (list[list[int]]): A list of secret bit strings.

    Returns: 
        array[float]: Probabilities for observing different outcomes.
    """
    #Do the Hadamard Transform
    ##### TODO #####

    # Apply the oracle we built
    ##### TODO #####

    #Redo the Hadamard Transform
    ##### TODO #####

    # Return the probabilies
    return qp.probs(wires=range(n_bits))

In [ ]:
def deutsch_jozsa(promise_var):
    """Implements the Deutsch–Jozsa algorithm and guesses the promise variable.
    
    Args:
        promise_var (int): Indicates whether the function is balanced (0) or constant (1).
        
    Returns: 
        int: A guess at the promise variable.
    """
    if promise_var == 0:
        how_many = 2**(n_bits - 1)
    else:
        how_many = np.random.choice([0, 2**n_bits]) 

    combos = multisol_combo(n_bits, how_many) # Generate random combinations
    zero_prob = circuit(combos)[0] # Probability of observing the all-zero state
    guess = int(np.rint(zero_prob)) # Round to nearest integer
    return guess

In [ ]:
deutsch_jozsa(1)

In [ ]:
how_many = 2**(n_bits - 1)
combos = multisol_combo(n_bits, how_many) # Generate random combinations
qp.draw_mpl(circuit,style = "pennylane")(combos)

In [ ]:
import itertools

n_bits = 3
dev = qp.device("default.qubit", wires=n_bits)

def oracle_from_function(f):
    """Phase oracle U_f |x> = (-1)^f(x) |x>."""
    inputs = itertools.product([0, 1], repeat=n_bits)  # |00..0>, |00..1>, ... in basis order
    return np.diag([(-1) ** f(x) for x in inputs])

@qp.qnode(dev)
def hoh_circuit(U):
    ##### TODO #####
    return qp.probs(wires=range(n_bits))

def deutsch_jozsa(f):
    zero_prob = hoh_circuit(oracle_from_function(f))[0]
    return "constant" if np.isclose(zero_prob, 1) else "balanced"

# Examples
print(deutsch_jozsa(lambda x: 0))             # constant
print(deutsch_jozsa(lambda x: 1))             # constant
print(deutsch_jozsa(lambda x: x[0]))          # balanced
print(deutsch_jozsa(lambda x: sum(x) % 2))    # balanced